# Question 1: do some vampires bite much more than others?

MATH 1660, November 2026.

Every vampire could bite for five days, and the number of people each one bit varies from vampire to vampire. Is that variation what we would expect if every vampire bit at the same rate, and some were just luckier than others? Or did some vampires bite at a higher rate than others? This is the question of Lloyd-Smith and others that we covered in class: one rate for every vampire (the **Poisson** model) against a rate for each vampire (the **negative binomial** model, whose dispersion $k$ is small when vampires differ a lot).

**How this connects to *does branching theory describe our game below the threshold?*** A branching process assumes that every vampire's number of bites is drawn from the same distribution, called the **offspring distribution**, independently of every other vampire's, whoever bit them and whenever they turned. The Poisson and the negative binomial are two candidates for that one distribution. (In the negative binomial, each vampire's own rate is drawn at random from the same gamma distribution, so every vampire's count still comes from the same distribution.) So this notebook takes the branching assumption as given and asks which offspring distribution fits our counts. It does not test the assumption itself. Question 2 tests one part of it: whether converts' counts come from the same distribution as roots'.

**Use practice mode to write your plan.** The data from our game are in the **research export**: a set of CSV files made by the game's platform, recording who played, who bit whom and when, and when each player turned and burned out. Each team is given a web link from which this notebook can read those files, but only after the team's plan is submitted.

Until then, the notebook does not read our game's data. While `PRACTICE = True` in Step 1, it makes up an export for an imaginary game, with the same files and the same columns, and runs every step on that. Use this to learn what each step computes: run the notebook, change the practice settings in Step 1, and run it again. Then decide the choices in Step 1 that your plan will state, such as `SITTING_MINUTES`.

After your plan is submitted, you are given the link. Paste it into Step 1, set `PRACTICE = False`, and run the notebook again: it now reads our game's export, and uses the choices your plan states.

**The result from the choices in your plan is your test.** Suppose that after seeing it you run the notebook again with a different choice, for example a different `SITTING_MINUTES` in Step 1. The result of that second run is **exploratory**, and your report says so. It gives the result from your plan's choices first. Then it gives the second result, labeled *exploratory*, with the choice you changed and why. An exploratory result is not a test. Its p-value is not the chance that the test describes, because the choice was made after seeing the data, and a choice made that way can be the one that happens to give the smallest p-value. An exploratory result can still suggest a question worth testing in another game.

Nobody has checked this notebook except its author. If a number looks wrong, it may be.

## How to use this notebook

1. **Make your own copy first:** *File*, then *Save a copy in Drive*.
2. Run the cells in order, from the top: click inside a cell and press **Shift+Enter**.
3. Change only the values in Step 1.

## Step 1: your choices, and the export

- `PRACTICE`: while it is `True`, the notebook makes up an export for an imaginary game and does not read our game's export. `PRACTICE_SEED` picks which made-up export you get; change it to get another.
- `PRACTICE_RATES`: how often the made-up vampires bite.
  - `"same"`: every vampire bites at the same rate, so the null hypothesis of Steps 5 and 6 (the Poisson model) is true. A test at the 0.05 level then rejects it in at most about 1 made-up export in 20; often fewer, because the counts take only whole values.
  - `"different"`: each vampire has their own rate, and a few vampires have most of it (negative binomial, $k = 0.2$).
- `PRACTICE_SITTINGS`: whether the made-up bites come in **sittings**.
  - `False`: every bite is on its own, at a random time in the vampire's lifetime. A sitting is then almost always one bite, and the two versions of Step 3 are almost always the same.
  - `True`: the rate set by `PRACTICE_RATES` is the rate at which a vampire starts sittings, and each sitting is 1, 2, or 3 bites a few minutes apart, as when friends scan one vampire's code one after another. With `"same"`, the number of sittings then follows the Poisson model, but the number of bites does not, because one sitting can hold several bites. So the tests reject the Poisson model for bites counted one by one far more often than for sittings, although every vampire has the same rate.

  Start with `"same"` and `False`, and check that small p-values are rare. Then try each of the other three combinations, and see what each test says about bites counted one by one and about sittings.
- `EXPORT_LINK`: empty until your plan is submitted. Then paste your team's export link between the quotes. It ends with `/`. **Do not share a notebook that has the link in it**: anyone with the link can read the research data. If you would rather upload the files, leave the link empty and upload `settings.csv`, `participants.csv`, `transmissions.csv`, and `state_changes.csv` with the folder icon on the left.
- `SITTING_MINUTES`: a bite that comes no more than this many minutes after the same vampire's previous bite belongs to the same sitting (Step 3). Rulebook R4.5 says that bites at one sitting are flagged and the analysis is run with and without them, and leaves the number of minutes to the analysis. Your plan states it.
- `ROOTS_ONLY`: `False` uses every finished vampire. `True` uses only finished **roots**, the vampires nobody bit (`is_seed` is `True`): a check, since converts may bite differently (that is question 2).
- `REPS`: how many repetitions each simulation makes.

In [ ]:
# Step 1: your choices
PRACTICE = True           # True: a made-up export. False: our export (only after your plan is submitted)
PRACTICE_SEED = 1         # change it to get a different made-up export
PRACTICE_RATES = "same"   # "same" or "different": how often the made-up vampires bite
PRACTICE_SITTINGS = False # False: every made-up bite on its own. True: bites in sittings of 1 to 3
EXPORT_LINK = ""          # after your plan is submitted: paste your team's export link here
SITTING_MINUTES = 10      # the value in your plan
ROOTS_ONLY = False        # False: every finished vampire. True: finished roots only
REPS = 10000              # repetitions of each simulation

Run the next five cells. The first four copy three files of the course simulator, `vampsim`, into this session, as of November 2026; you do not need to read them. The tests below call two things from them: `chains.offspring_lrt`, the likelihood-ratio statistic of Step 6, and `model.poisson`, which draws a number from the Poisson model. The fifth reads the export, or makes one up.

In [ ]:
# Step 1b: make a folder for the simulator files
import os
os.makedirs("vampsim", exist_ok=True)
open("vampsim/__init__.py", "w").close()
print("Folder ready.")

In [ ]:
%%writefile vampsim/model.py
"""The game's rules as an event-driven stochastic simulation.

One call to simulate() runs one outbreak and returns an Outbreak record holding
every scan: its time, the vampire whose code it was, the person who scanned,
whether the two share a residence hall, and the outcome. Everything else
(daily counts, final size, offspring, ask histories for P6) is computed from
that record, so a new analysis never needs a change to the simulation.

Rules as modeled (see README.md for what is left out):
- A person bitten at time t cannot bite during the incubation period T_E, then turns
  and bites for T_I days, then is removed for good.
- While biting, a vampire's scans arrive at random times at rate a_i per day.
- Each scan is by a person chosen at random from the vampire's own hall (with
  probability p_own) or from the other halls. A scan by someone already bitten
  or removed is lost.
- A mortal who scans says yes with a probability set by q, the hall label, and
  their own willingness. A yes is a bite, and starts their incubation.

Who is counted (defined 2026-10-11):
- N is the number of people the vampires' scans fall on.
- N_eff is the number of those people who would join if asked enough times
  during the game: N minus the never_frac share who always say no. People who
  are never asked are not in N at all. Neither group can join, so the outbreak
  depends on N_eff and not on N: a run with N people and never_frac = f behaves
  like a run with N_eff = (1 - f) N people and never_frac = 0, given the same
  beta (check 7). With never_frac = 0, the default, N is N_eff.
- With q_spread > 0, everyone keeps some chance of a yes, so all N are in N_eff,
  but the outbreak is smaller than a run with the same N_eff and no spread, and
  the final-size relation no longer holds exactly.
"""

from __future__ import annotations

import heapq
import math
import random
from dataclasses import dataclass, field
from typing import List, Optional


@dataclass
class Params:
    """Settings for one outbreak. Times are in days.

    beta, k and q describe the vampires and the campus:
      beta  mean successful bites per vampire-day in a fully mortal campus, a*q.
      k     dispersion of beta_i across vampires. Each vampire's scan rate a_i is
            drawn from a gamma distribution with mean a and shape k, so the number
            each vampire bites is negative binomial with dispersion k (Lloyd-Smith).
            None means every vampire has the same rate, which gives Poisson offspring.
      q     campus mean fraction of scans that end in a yes, over scans by
            mortals among all N people (the never_frac share included; walking
            away counts as no). Given beta and k, q
            changes nothing about the outbreak's size or timing; it sets how many
            scans (and so how many declines) it takes to produce the bites, which
            is what the ask histories and P6 depend on.

    N and N_eff: see the module docstring. With never_frac = 0, N is N_eff.
    """

    N: int = 500                      # people the scans fall on; N_eff if never_frac = 0
    n0: int = 15                      # seeds
    beta: float = 0.75                # bites per vampire-day; R0 = beta * TI
    k: Optional[float] = None         # dispersion of beta_i; None = homogeneous
    q: float = 0.25                   # mean yes per scan
    TE: float = 1.0                   # incubation period, days
    TI: float = 2.0                   # vampire lifetime after incubation, days
    t_end: float = 28.0               # the game closes; Oct 16 -> Nov 13

    # residence halls; None = one hall, which is homogeneous mixing
    halls: Optional[List[int]] = None  # hall sizes, summing to N
    pi_same: float = 0.7              # share of scans in the vampire's own hall
    steer: float = 0.0                # 0: pi_same fixed. >0: vampires shift to the
                                      # other halls as their own runs out of mortals
    seed_halls: Optional[int] = None  # seeds spread over this many random halls;
                                      # None = seeds are random people
    seeds_incubate: bool = False      # False: seeds are vampires at launch, as in
                                      # the game. True: seeds are bitten at launch
                                      # and wait T_E, as in the working notes
                                      # written before 2026-09-17
    rho: float = 1.0                  # cross-hall yes rate / same-hall yes rate

    # mortals' willingness; both keep the campus mean at q
    q_spread: float = 0.0             # half of mortals at q(1+c), half at q(1-c)
    never_frac: float = 0.0           # fraction of the N who never say yes; they
                                      # are outside N_eff. Seeds are drawn from
                                      # all N, so some may be such people

    linkage: float = 1.0              # chance a decline reaches the analysis

    stop_Z: Optional[int] = None      # stop once this many are bitten; for
                                      # early-phase questions on large campuses

    @classmethod
    def from_R0(cls, R0: float, **kw) -> "Params":
        TI = kw.get("TI", cls.TI)
        return cls(beta=R0 / TI, **kw)

    @property
    def N_eff(self) -> float:
        """Expected number of the N people who would join if asked enough:
        N (1 - never_frac). Equal to N when never_frac = 0."""
        return self.N * (1 - self.never_frac)

    @property
    def a(self) -> float:
        """Mean scans per vampire-day."""
        return self.beta / self.q

    @property
    def R0(self) -> float:
        """Mean number bitten by one vampire in a fully mortal campus."""
        return self.beta * self.TI

    @property
    def Tg(self) -> float:
        """Mean generation time, days."""
        return self.TE + self.TI / 2


@dataclass
class Scan:
    t: float
    vampire: int
    target: int
    same_hall: bool
    outcome: str        # "yes", "no", or "lost" (target already bitten)
    linked: bool        # a "no" that reaches the target's history; "yes" always


@dataclass
class Outbreak:
    params: Params
    hall: List[int]
    bitten: List[Optional[float]]     # time bitten; seeds at -T_E (or 0.0 if
                                      # seeds_incubate), so they turn at launch;
                                      # None = never
    infector: List[Optional[int]]     # None for seeds and the never-bitten
    rate: List[Optional[float]]       # a_i for everyone bitten
    scans: List[Scan] = field(default_factory=list)

    # ---- counts ----
    @property
    def seeds(self) -> List[int]:
        return [j for j, t in enumerate(self.bitten) if t is not None and self.infector[j] is None]

    @property
    def Z(self) -> int:
        """Number ever bitten by the close, seeds included."""
        return sum(t is not None for t in self.bitten)

    def bite_times(self) -> List[float]:
        """Times of successful bites, seeds excluded, sorted."""
        return sorted(s.t for s in self.scans if s.outcome == "yes")

    def daily(self) -> List[int]:
        """Bites per day of the game, day 0 = launch."""
        days = [0] * int(math.ceil(self.params.t_end))
        for t in self.bite_times():
            days[min(int(t), len(days) - 1)] += 1
        return days

    def peak_day(self) -> Optional[int]:
        d = self.daily()
        return max(range(len(d)), key=lambda i: d[i]) if any(d) else None

    def day_quantile(self, frac: float = 0.95) -> Optional[float]:
        """Day by which frac of the bites have happened (d95 by default)."""
        b = self.bite_times()
        if not b:
            return None
        return b[max(0, math.ceil(frac * len(b)) - 1)]

    def active_at_close(self) -> int:
        """Vampires still biting when the game closes."""
        p = self.params
        return sum(1 for t in self.bitten
                   if t is not None and t + p.TE + p.TI > p.t_end)

    def offspring(self, complete_only: bool = True) -> List[int]:
        """Number each vampire bit. With complete_only, only vampires whose
        lifetime ended before the close, so no count is cut short."""
        p = self.params
        n = [0] * len(self.bitten)
        for s in self.scans:
            if s.outcome == "yes":
                n[s.vampire] += 1
        return [n[j] for j, t in enumerate(self.bitten)
                if t is not None and (not complete_only or t + p.TE + p.TI <= p.t_end)]

    def scan_counts(self) -> dict:
        c = {"yes": 0, "no": 0, "lost": 0}
        for s in self.scans:
            c[s.outcome] += 1
        return c


def poisson(lam: float, rng: random.Random) -> int:
    n = 0
    while lam > 30.0:                  # split to keep exp(-lam) away from underflow
        n += poisson(30.0, rng)
        lam -= 30.0
    L, k, prod = math.exp(-lam), 0, 1.0
    while True:
        prod *= rng.random()
        if prod <= L:
            return n + k
        k += 1


def simulate(p: Params, seed: Optional[int] = None,
             rng: Optional[random.Random] = None) -> Outbreak:
    rng = rng or random.Random(seed)
    sizes = list(p.halls) if p.halls else [p.N]
    if sum(sizes) != p.N:
        raise ValueError("hall sizes must sum to N")
    nh = len(sizes)
    hall, members, start = [], [], 0
    for h, s in enumerate(sizes):
        hall += [h] * s
        members.append(list(range(start, start + s)))
        start += s
    N = p.N

    # willingness factor, mean 1, so the campus mean yes rate stays q
    willing = 1.0 / (1.0 - p.never_frac) if p.never_frac < 1 else 0.0
    f = []
    for _ in range(N):
        if rng.random() < p.never_frac:
            f.append(0.0)
        else:
            f.append(willing * (1 + p.q_spread if rng.random() < 0.5 else 1 - p.q_spread))

    # same- and cross-hall yes rates, scaled so the mean over asks at pi_same is q
    if nh > 1:
        q_same = p.q / (p.pi_same + (1 - p.pi_same) * p.rho)
        q_cross = p.rho * q_same
    else:
        q_same = q_cross = p.q

    ob = Outbreak(p, hall, [None] * N, [None] * N, [None] * N)
    mortal = sizes[:]                  # mortals left in each hall
    n_mortal = N
    heap: list = []
    seq = 0

    def bite(j: int, t: float, by: Optional[int]) -> None:
        nonlocal seq, n_mortal
        ob.bitten[j], ob.infector[j] = t, by
        mortal[hall[j]] -= 1
        n_mortal -= 1
        ai = p.a if p.k is None else rng.gammavariate(p.k, p.a / p.k)
        ob.rate[j] = ai
        for _ in range(poisson(ai * p.TI, rng)):
            ts = t + p.TE + rng.random() * p.TI
            if ts <= p.t_end:
                heapq.heappush(heap, (ts, seq, j))
                seq += 1

    # seeds
    if p.seed_halls is None or nh == 1:
        seeds = rng.sample(range(N), p.n0)
    else:
        chosen = rng.sample(range(nh), min(p.seed_halls, nh))
        pools = {h: rng.sample(members[h], len(members[h])) for h in chosen}
        seeds, i = [], 0
        while len(seeds) < p.n0:
            h = chosen[i % len(chosen)]
            if pools[h]:
                seeds.append(pools[h].pop())
            elif not any(pools.values()):
                raise ValueError("more seeds than people in the seeded halls")
            i += 1
    for s in seeds:
        bite(s, 0.0 if p.seeds_incubate else -p.TE, None)

    while heap:
        t, _, v = heapq.heappop(heap)
        h = hall[v]
        if nh == 1:
            own = True
        else:
            w_h = mortal[h] / sizes[h]
            w_o = (n_mortal - mortal[h]) / (N - sizes[h])
            num = p.pi_same * w_h ** p.steer
            den = num + (1 - p.pi_same) * w_o ** p.steer
            own = rng.random() < (num / den if den > 0 else p.pi_same)
        while True:
            if own:
                tgt = rng.choice(members[h])
            else:
                tgt = rng.randrange(N)
                if hall[tgt] == h:
                    continue
            if tgt != v:
                break
        same = hall[tgt] == h
        if ob.bitten[tgt] is not None:
            ob.scans.append(Scan(t, v, tgt, same, "lost", False))
            continue
        pr = min(1.0, (q_same if same else q_cross) * f[tgt])
        if rng.random() < pr:
            ob.scans.append(Scan(t, v, tgt, same, "yes", True))
            bite(tgt, t, v)
            if p.stop_Z is not None and N - n_mortal >= p.stop_Z:
                break
        else:
            ob.scans.append(Scan(t, v, tgt, same, "no", rng.random() < p.linkage))
    return ob


def run_many(p: Params, n: int, seed: int = 0) -> List[Outbreak]:
    rng = random.Random(seed)
    return [simulate(p, rng=rng) for _ in range(n)]

In [ ]:
%%writefile vampsim/theory.py
"""Closed-form results the simulation should agree with.

Each function here answers a question without simulating, under the
assumptions of a large, well-mixed campus. checks.py compares them with
simulate(); where they disagree, the simulation is the game's rules and the
formula is the approximation.
"""

from __future__ import annotations

import math
from typing import Optional


def extinction(R0: float, k: Optional[float] = None, n0: int = 1) -> float:
    """Chance an outbreak from n0 seeds dies out, in a campus too large to run
    out of mortals. Offspring are negative binomial with mean R0 and dispersion
    k, or Poisson when k is None. Solves x = G(x) for the smallest root."""
    if R0 <= 1:
        return 1.0
    if k is None:
        G = lambda x: math.exp(-R0 * (1 - x))
    else:
        G = lambda x: (1 + R0 * (1 - x) / k) ** (-k)
    x = 0.0
    for _ in range(100000):
        nx = G(x)
        if abs(nx - x) < 1e-13:
            break
        x = nx
    return x ** n0


def final_size(R0: float, s0: float = 1.0) -> float:
    """Fraction never bitten, s_inf, from s_inf = s0 exp(-R0 (1 - s_inf)).
    Holds for any spread in beta_i across vampires and any incubation or
    lifetime, so long as mixing is uniform and willingness is the same for all."""
    s = 1e-12
    for _ in range(100000):
        ns = s0 * math.exp(-R0 * (1 - s))
        if abs(ns - s) < 1e-14:
            break
        s = ns
    return s


def M(r: float, TE: float, TI: float) -> float:
    """Laplace transform of the generation interval, uniform on [TE, TE+TI]."""
    if abs(r) < 1e-12:
        return 1.0
    return math.exp(-r * TE) * (1 - math.exp(-r * TI)) / (r * TI)


def growth_rate(R0: float, TE: float, TI: float) -> float:
    """Early growth rate per day, the real root of 1 = R0 M(r)."""
    if R0 == 1:
        return 0.0
    lo, hi = (1e-9, 20.0) if R0 > 1 else (-20.0, -1e-9)
    for _ in range(200):
        mid = (lo + hi) / 2
        if R0 * M(mid, TE, TI) > 1:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2


def doubling_time(R0: float, TE: float, TI: float) -> float:
    return math.log(2) / growth_rate(R0, TE, TI)


# Chains below the threshold (added 2026-10-25). A chain is one root and everyone
# descended from it; its size Y counts the root. Offspring are negative binomial
# with mean R and dispersion k, or Poisson when k is None, on a campus too large
# to run out of mortals.

def offspring_pgf(R: float, k: Optional[float] = None):
    """G(s), the generating function of the number one vampire bites."""
    if k is None:
        return lambda s: math.exp(R * (s - 1))
    return lambda s: (1 + R * (1 - s) / k) ** (-k)


def mean_chain_size(R: float) -> float:
    """E[Y] = 1 + R + R^2 + ... = 1/(1 - R); the same for every k."""
    return 1 / (1 - R) if R < 1 else math.inf


def chain_size_var(R: float, k: Optional[float] = None) -> float:
    """Var(Y) = sigma^2 / (1 - R)^3, with sigma^2 = R + R^2/k the offspring variance."""
    if R >= 1:
        return math.inf
    s2 = R if k is None else R + R * R / k
    return s2 / (1 - R) ** 3


def chain_size_pmf(y: int, R: float, k: Optional[float] = None) -> float:
    """P(Y = y) = (1/y) P(X_1 + ... + X_y = y - 1), the hitting-time theorem.
    Poisson: the Borel distribution. Negative binomial: Blumberg and Lloyd-Smith (2013)."""
    if y < 1:
        return 0.0
    if R == 0:
        return 1.0 if y == 1 else 0.0
    if k is None:
        return math.exp(-y * R + (y - 1) * math.log(y * R) - math.lgamma(y + 1))
    logp = (math.lgamma(y * k + y - 1) - math.lgamma(y * k) - math.lgamma(y + 1)
            + (y - 1) * math.log(R / k) - (y * k + y - 1) * math.log(1 + R / k))
    return math.exp(logp)


def generations_cdf(g: int, R: float, k: Optional[float] = None) -> float:
    """P(a chain has at most g generations of bites), g = 0 when the root bit no one.
    Equal to G applied g + 1 times to 0."""
    G, x = offspring_pgf(R, k), 0.0
    for _ in range(g + 1):
        x = G(x)
    return x

In [ ]:
%%writefile vampsim/chains.py
"""Whether chains below the threshold show that a few vampires do most of the
biting, and how many chains it takes to tell.

Added 2026-10-25 for the team question *does branching theory describe our game
below R0 = 1?* Below the threshold every chain ends, and branching theory gives
the whole distribution of chain sizes from R and the dispersion k
(theory.chain_size_pmf). The mean is 1/(1 - R) for every k, so only the shape
can tell k: with small k there are more chains of the seed alone and a longer
tail. The question here is how many finished chains that takes.

Both models are branching models: Poisson gives every vampire the same R,
negative binomial gives each vampire their own R_i, gamma with mean R and
shape k. So these tests measure how much vampires differ inside branching
theory; they do not test branching theory itself (clarified 2026-10-26).

Two tests of Poisson (every vampire bites at the same rate) against negative
binomial (small k), each a likelihood-ratio test with R at its estimate:

- sizes: on the size of each chain alone. R is estimated as 1 - n / (total
  people), which is its maximum likelihood estimate for every k.
- offspring: on each vampire's bite count. The export records every bite, so
  this uses all of the tree, of which the chain sizes are a summary.

Critical values come from chains generated under Poisson at the same R and
number of chains, not from the chi-squared approximation, which is poor at
these sizes. sample_chain() generates chains by the branching process the
theory assumes; check 10 in checks.py compares it with theory.chain_size_pmf,
and check 8 compares simulate() with the same formula.
"""

from __future__ import annotations

import math
import random
from collections import Counter
from typing import List, Optional, Sequence, Tuple

from . import theory
from .model import poisson

K_GRID = [math.exp(math.log(0.01) + (math.log(1000.0) - math.log(0.01)) * i / 80)
          for i in range(81)]


def sample_chain(R: float, k: Optional[float], rng: random.Random,
                 cap: int = 10000) -> Tuple[int, List[int]]:
    """One chain from one root: its size, and the bite count of each member.
    Each vampire bites Poisson(lambda_i) people, lambda_i = R, or gamma with
    mean R and shape k. Stops at cap members (only possible near R = 1)."""
    offspring: List[int] = []
    waiting = 1
    while waiting and len(offspring) < cap:
        lam = R if k is None else rng.gammavariate(k, R / k)
        n = poisson(lam, rng)
        offspring.append(n)
        waiting += n - 1
    return len(offspring), offspring


def sizes_lrt(sizes: Sequence[int]) -> Tuple[float, float]:
    """Likelihood-ratio statistic for k against Poisson, from chain sizes, and
    the k that maximizes the likelihood (inf when Poisson fits best)."""
    n, total = len(sizes), sum(sizes)
    R = 1 - n / total
    if R <= 0:
        return 0.0, math.inf
    counts = Counter(sizes)

    def ll(k):
        return sum(c * math.log(theory.chain_size_pmf(y, R, k)) for y, c in counts.items())

    ll0 = ll(None)
    best, kbest = ll0, math.inf
    for k in K_GRID:
        v = ll(k)
        if v > best:
            best, kbest = v, k
    return 2 * (best - ll0), kbest


def offspring_lrt(counts: Sequence[int]) -> Tuple[float, float]:
    """Likelihood-ratio statistic for k against Poisson, from bite counts, and
    the k that maximizes the likelihood (inf when Poisson fits best)."""
    m = sum(counts) / len(counts)
    if m == 0:
        return 0.0, math.inf
    tally = Counter(counts)
    ll0 = sum(c * (y * math.log(m) - m - math.lgamma(y + 1)) for y, c in tally.items())

    def ll(k):
        return sum(c * (math.lgamma(y + k) - math.lgamma(k) - math.lgamma(y + 1)
                        + k * math.log(k / (k + m)) + y * math.log(m / (k + m)))
                   for y, c in tally.items())

    best, kbest = ll0, math.inf
    for k in K_GRID:
        v = ll(k)
        if v > best:
            best, kbest = v, k
    return 2 * (best - ll0), kbest


def singles(sizes: Sequence[int]) -> float:
    """The simple statistic: the number of chains that are the root alone, less
    the number Poisson expects at the estimated R, n exp(-R). Positive when there
    are more lone roots than Poisson allows, as with small k."""
    n = len(sizes)
    R = 1 - n / sum(sizes)
    return sum(s == 1 for s in sizes) - n * math.exp(-R)


def zeros(counts: Sequence[int]) -> float:
    """The same simple statistic on bite counts: the number of vampires who bit
    no one, less the number Poisson expects at the mean, n exp(-mean)."""
    n = len(counts)
    return sum(x == 0 for x in counts) - n * math.exp(-sum(counts) / n)


def _stat(test: str, chains: List[Tuple[int, List[int]]]) -> float:
    if test == "sizes":
        return sizes_lrt([s for s, _ in chains])[0]
    if test == "singles":
        return singles([s for s, _ in chains])
    counts = [x for _, off in chains for x in off]
    if test == "zeros":
        return zeros(counts)
    return offspring_lrt(counts)[0]


def critical_value(test: str, n_chains: int, R: float, alpha: float = 0.05,
                   reps: int = 2000, rng: Optional[random.Random] = None) -> float:
    """The statistic that Poisson chains exceed with chance alpha, at this R and
    number of chains."""
    rng = rng or random.Random()
    null = sorted(_stat(test, [sample_chain(R, None, rng) for _ in range(n_chains)])
                  for _ in range(reps))
    return null[int(math.ceil((1 - alpha) * reps)) - 1]


def power(test: str, n_chains: int, R: float, k: float, alpha: float = 0.05,
          reps: int = 2000, rng: Optional[random.Random] = None) -> float:
    """Chance the test rejects Poisson at level alpha when the true dispersion
    is k, from n_chains finished chains at R. test is "sizes" or "offspring"
    (likelihood ratio), or "singles" or "zeros" (the simple statistics)."""
    rng = rng or random.Random()
    crit = critical_value(test, n_chains, R, alpha, reps, rng)
    hits = sum(_stat(test, [sample_chain(R, k, rng) for _ in range(n_chains)]) > crit
               for _ in range(reps))
    return hits / reps

In [ ]:
# Step 1c: read the export, or make one up
import csv, io, math, random, urllib.request
from datetime import datetime, timedelta, timezone

FILES = ["settings.csv", "participants.csv", "transmissions.csv", "state_changes.csv"]

def read(name):
    if EXPORT_LINK:
        text = urllib.request.urlopen(EXPORT_LINK + name).read().decode("utf-8")
    else:
        text = open(name, encoding="utf-8").read()
    return list(csv.DictReader(io.StringIO(text)))

t = lambda s: datetime.fromisoformat(s) if s else None
hours = lambda a, b: (b - a).total_seconds() / 3600     # hours from a to b
iso = lambda d: d.isoformat()

def show_p(p, reps=None):
    # a p-value with 4 decimals, a very small one in powers of 10, and a simulated 0 as "below 1/reps"
    if reps and p == 0:
        return f"0 (below 1/{reps})"
    return f"{p:.4f}" if p >= 0.001 else f"{p:.2e}"
LAUNCH = datetime(2026, 10, 16, 23, 0, tzinfo=timezone.utc)   # used only by the made-up export
SETTINGS = {"spree_start_at": "2026-11-09T18:03:00+00:00", "end_at": "2026-11-14T00:00:00+00:00",
            "refreshed_at": "2026-11-14T00:00:00+00:00"}
import importlib, vampsim.model, vampsim.theory, vampsim.chains
for module in (vampsim.model, vampsim.theory, vampsim.chains):
    importlib.reload(module)
from vampsim import chains
from vampsim.model import poisson

def made_up_export(seed):
    # 50 vampires, each with a 120-hour lifetime that ends before the spree; 30 are roots.
    # Each vampire starts sittings at random times in their lifetime. With PRACTICE_SITTINGS
    # False a sitting is one bite, 0.3 sittings each on average; with True it is 1, 2, or 3
    # bites, 1 to 4 minutes apart, 0.18 sittings each on average. Either way a vampire makes
    # about 0.3 bites on average.
    rng = random.Random(seed)
    out = {"settings.csv": [dict(SETTINGS)], "participants.csv": [], "transmissions.csv": [],
           "state_changes.csv": []}
    for i in range(50):
        pid = f"practice-{i + 1:02d}"
        root = i < 30
        turn = LAUNCH + timedelta(hours=rng.uniform(0, 17 * 24))
        out["participants.csv"].append({"participant_id": pid, "is_seed": str(root),
                                        "joined_through": "walker" if root else "",
                                        "joined_at": iso(turn if root else turn - timedelta(hours=12)),
                                        "turn_at": iso(turn)})
        out["state_changes.csv"].append({"participant_id": pid, "change": "vampire → burned out",
                                         "at": iso(turn + timedelta(hours=120))})
        mean = 0.18 if PRACTICE_SITTINGS else 0.3        # sittings per vampire, on average
        rate = mean if PRACTICE_RATES == "same" else rng.gammavariate(0.2, mean / 0.2)
        for _ in range(poisson(rate, rng)):
            at = turn + timedelta(hours=rng.uniform(0, 119))
            for j in range(rng.choice([1, 1, 1, 2, 2, 3]) if PRACTICE_SITTINGS else 1):
                if j > 0:
                    at += timedelta(minutes=rng.uniform(1, 4))
                out["transmissions.csv"].append({"source_id": pid, "target_id": f"{pid}-bite-{len(out['transmissions.csv'])}",
                                                 "at": iso(at), "target_already_in_game": "False"})
    return out

data = made_up_export(PRACTICE_SEED) if PRACTICE else {name: read(name) for name in FILES}
settings = data["settings.csv"][0]
players = {p["participant_id"]: p for p in data["participants.csv"]}
rows = data["transmissions.csv"]
if PRACTICE:
    print(f"PRACTICE MODE: a made-up export (seed {PRACTICE_SEED}). None of these numbers comes from our game.")
else:
    print("Our export, made at", settings["refreshed_at"], "(UTC)")
print(len(players), "players,", len(rows), "rows in transmissions.csv")

## Step 2: finished vampires, and their bites

A vampire is **finished** when their five days are over, so their number of bites can no longer change. The spree, which started at `spree_start_at` in `settings.csv`, let burned-out players bite again, so a vampire counts as finished here only if they burned out *before the spree started*. Use `state_changes.csv` for this, not the `burnout_at` column of `participants.csv`: for players the spree brought back, `burnout_at` shows the end of the game, not the time they first burned out.

A **bite** is a row of `transmissions.csv` that made its target a new player (`target_already_in_game` is `False`). A row with `True` is a scan of someone already in the game, and is not a bite. A finished vampire's bites are the ones they made **before their first burnout**; a bite made during the spree, after the spree brought them back, is left out.

In [ ]:
# Step 2: finished vampires, and the bites each made before their first burnout
spree = t(settings["spree_start_at"])
first_burnout = {}
for row in data["state_changes.csv"]:
    if row["change"] == "vampire → burned out":
        first_burnout.setdefault(row["participant_id"], t(row["at"]))

finished = [p for p in players.values()
            if p["participant_id"] in first_burnout and first_burnout[p["participant_id"]] < spree]
finished_ids = {p["participant_id"] for p in finished}

bites_by = {}      # participant_id -> times of their bites before their first burnout
recruiter = {}     # participant_id of a convert -> participant_id of the vampire who bit them
for b in rows:
    if b["target_already_in_game"] != "False":
        continue
    recruiter[b["target_id"]] = b["source_id"]
    src = b["source_id"]
    if src in first_burnout and t(b["at"]) < first_burnout[src]:
        bites_by.setdefault(src, []).append(t(b["at"]))

def bites_of(pid):
    return len(bites_by.get(pid, []))

print(len(finished), "finished vampires;",
      sum(bites_of(p["participant_id"]) > 0 for p in finished), "of them bit someone;",
      sum(bites_of(p["participant_id"]) for p in finished), "bites between them")

## Step 3: the counts

The vampires in this question are the finished vampires, or the finished roots if `ROOTS_ONLY` is `True`. Number them $1, 2, \dots, n$. For vampire $i$:

- $y_i$ is the number of **bites** vampire $i$ made: the number of people they made into new players, before their first burnout.
- $s_i$ is the number of **sittings**. Put vampire $i$'s bites in order of time. Their first bite starts a sitting. Each later bite starts a new sitting if it came more than `SITTING_MINUTES` minutes after vampire $i$'s previous bite; otherwise it belongs to the sitting already going.

*Example,* with `SITTING_MINUTES = 10`: bites at 9:00, 9:02, and 9:30 pm. The bite at 9:02 is 2 minutes after the one at 9:00, so it is in the same sitting; the bite at 9:30 is 28 minutes after the one at 9:02, so it starts a new one. So $y_i = 3$ and $s_i = 2$.

Then, for the whole group:

- $B = y_1 + y_2 + \dots + y_n$, the total number of bites, and $S = s_1 + \dots + s_n$, the total number of sittings.
- $R = B / n$, the average number of bites per vampire.
- $D$, the number of **biters**: vampires with $y_i \ge 1$. A vampire with at least one bite has at least one sitting, so $D$ is the same in both versions.

Every test below runs twice: once on the bite counts $y_1, \dots, y_n$ (*bites counted one by one*), and once on the sitting counts $s_1, \dots, s_n$ (*sittings*). The cell prints how many vampires have each count.

In [ ]:
# Step 3: the bite counts y_i and the sitting counts s_i
group = [p for p in finished if not ROOTS_ONLY or p["is_seed"] == "True"]

def sittings(times):
    count, previous = 0, None
    for a in sorted(times):
        if previous is None or hours(previous, a) * 60 > SITTING_MINUTES:
            count += 1           # this bite starts a new sitting
        previous = a
    return count

y = [bites_of(p["participant_id"]) for p in group]
s = [sittings(bites_by.get(p["participant_id"], [])) for p in group]
versions = {"bites counted one by one": y, "sittings": s}

n, D = len(y), sum(x >= 1 for x in y)
print(f"n = {n} vampires ({'finished roots only' if ROOTS_ONLY else 'every finished vampire'})")
print(f"B = {sum(y)} bites, S = {sum(s)} sittings (SITTING_MINUTES = {SITTING_MINUTES}), D = {D} biters")
print(f"R = B / n = {sum(y) / n:.3f} bites per vampire")
print()
print(f"{'count':>5}  {'vampires with this many bites':>30}  {'vampires with this many sittings':>33}")
for c in range(max(y) + 1):
    print(f"{c:>5}  {y.count(c):>30}  {s.count(c):>33}")

## Step 4: $R$, and how sure we are of it

$R$ is the first number a report gives: the average number of people a finished vampire bit. Its interval comes from **resampling the vampires**, as on October 30: draw $n$ vampires at random from our $n$, with replacement (so a vampire can be drawn more than once), and take the average of their counts. Do that `REPS` times. The middle 95 percent of those averages is the interval.

In [ ]:
# Step 4: R, with an interval from resampling the vampires
rng = random.Random(4)
for name, counts in versions.items():
    means = sorted(sum(rng.choice(counts) for _ in range(n)) / n for _ in range(REPS))
    low, high = means[int(0.025 * REPS)], means[int(0.975 * REPS) - 1]
    print(f"{name}: average {sum(counts) / n:.3f} per vampire, 95% interval {low:.3f} to {high:.3f}")

## Step 5: the shuffle test

### 5a. What the test asks

**The null hypothesis:** every vampire bites at the same rate, independently of the others (the Poisson model).

**What that says once the total is known.** If every vampire bites at the same rate, then once we know that there were $B$ bites in all, each of the $B$ bites is equally likely to be any vampire's, independently of the other bites. (This is a property of the Poisson model: given their total, the counts are like $B$ balls dropped one at a time, each into one of $n$ boxes chosen at random.) So the test keeps the total $B$ and hands the bites out again at random.

**The statistic** is $D$, the number of biters (Step 3).

**Why a small $D$ is the sign.** If a few vampires do most of the biting, the bites pile up on those few, and fewer vampires have any bites than a random hand-out gives. Write $D_{\text{obs}}$ for the observed value of $D$: the value Step 3 computed from the export this notebook read (in practice mode, the made-up export). The p-value is the chance, under the null hypothesis, that a random hand-out gives $D$ **equal to $D_{\text{obs}}$ or smaller**:

$$p = \Pr(D \le D_{\text{obs}}).$$ Larger values of $D$ are not counted, because they would point the other way.

The p-value is **conditional on the total**: it compares our counts only with hand-outs of the same $B$ bites. In the sittings version, the total is $S$, the number of sittings, and the hand-out is of $S$ sittings.

### 5b. Computing the chance of each $D$ exactly

Hand the $B$ bites out one at a time, and keep track of $d$, the number of vampires who have at least one bite so far.

- The first bite makes $d = 1$.
- Each later bite goes to one of the $d$ vampires who already have a bite with chance $d/n$, and then $d$ stays the same; or it goes to one of the other $n - d$ vampires with chance $(n - d)/n$, and then $d$ goes up by 1.

So, from the chances of each $d$ after some bites, the chances after one more bite are

$$\Pr(\text{after: } d) = \Pr(\text{before: } d)\,\frac{d}{n} + \Pr(\text{before: } d - 1)\,\frac{n - d + 1}{n} .$$

*Example,* $n = 3$ vampires and $B = 2$ bites. After the first bite, $d = 1$. The second bite goes to the same vampire with chance $1/3$, so $\Pr(D = 1) = 1/3$, or to another with chance $2/3$, so $\Pr(D = 2) = 2/3$. If our $D$ were 1, the p-value would be $\Pr(D \le 1) = 1/3$.

The cell does this for all $B$ bites and prints every value of $D$ with its chance. The row marked with an arrow is $D_{\text{obs}}$; the last column on that row is the p-value. The average of $D$ under the null hypothesis is also printed: it is $n\,(1 - (1 - 1/n)^B)$.

In [ ]:
# Step 5b: the exact chance of each D, and the p-value, for each version
def biters_distribution(n, total):
    dist = [1.0] + [0.0] * n        # dist[d] = chance that d vampires have a bite; before any bite, d = 0
    for _ in range(total):
        new = [0.0] * (n + 1)
        for d, pd in enumerate(dist):
            if pd == 0:
                continue
            new[d] += pd * d / n                    # the bite goes to a vampire who already has one
            if d < n:
                new[d + 1] += pd * (n - d) / n      # the bite goes to a vampire who had none
        dist = new
    return dist

shuffle_p = {}
for name, counts in versions.items():
    total = sum(counts)
    dist = biters_distribution(n, total)
    shuffle_p[name] = p = sum(dist[:D + 1])
    print(f"--- {name}: n = {n}, total = {total}, observed D = {D}")
    print(f"{'D':>5}  {'chance D is exactly this':>25}  {'chance D is this or smaller':>28}")
    for d in range(min(n, total) + 1):
        if total > 0 and d == 0:
            continue
        mark = "   <- observed D" if d == D else ""
        print(f"{d:>5}  {dist[d]:>25.2e}  {sum(dist[:d + 1]):>28.2e}{mark}")
    print(f"Average of D if every vampire bites at the same rate: {n * (1 - (1 - 1 / n) ** total):.2f}. Observed D = {D}.")
    print(f"p-value, the chance of {D} or fewer biters: {show_p(p)}")
    print()

### 5c. Check by simulation

Do what the null hypothesis says, many times, and count: hand the bites out at random, one at a time, each to one of the $n$ vampires chosen with equal chance; count the vampires who got at least one; repeat `REPS` times. The share of repetitions with $D$ equal to $D_{\text{obs}}$ or smaller should be close to the p-value of 5b. If it is not, something in 5b is wrong. When the p-value is very small, the simulation may find no repetition with $D$ that small; that shows only that $p$ is below $1/\text{REPS}$.

In [ ]:
# Step 5c: the shuffle, simulated
rng = random.Random(5)
for name, counts in versions.items():
    total = sum(counts)
    hits = sum(len({rng.randrange(n) for _ in range(total)}) <= D for _ in range(REPS))
    print(f"{name}: share of {REPS} random hand-outs with {D} or fewer biters: {show_p(hits / REPS, REPS)}"
          f"   (5b: {show_p(shuffle_p[name])})")

## Step 6: the likelihood-ratio test

The shuffle test uses only how many vampires bit. The likelihood-ratio test uses the whole count of each vampire: it also notices a vampire with an unusually large count.

### 6a. The two models

For a model that gives a vampire $y$ bites with chance $P(y)$, the **likelihood** of our counts is the chance the model gives to all of them together, $P(y_1)\,P(y_2) \cdots P(y_n)$. We work with its natural logarithm, which turns the product into a sum, and write it $\ell$. A model under which our counts were more likely has a larger $\ell$.

In both models, the average $R$ is set to the observed average, $B/n$ from Step 3.

- **Poisson**, every vampire with the same average $R$: $P(y) = e^{-R} R^y / y!$. Its log-likelihood is $\ell_{\text{P}}$.
- **Negative binomial**, average $R$ and dispersion $k$: $P(y) = \dfrac{\Gamma(y + k)}{\Gamma(k)\, y!} \left(\dfrac{k}{k + R}\right)^{k} \left(\dfrac{R}{k + R}\right)^{y}$. Here $\Gamma$ is the gamma function, which for whole numbers is $\Gamma(m) = (m - 1)!$. A small $k$ means vampires differ a lot; as $k$ grows, the model comes closer and closer to Poisson. Its log-likelihood is $\ell_{\text{NB}}(k)$.

### 6b. The statistic

$$\Lambda = 2\left[\max_k \ell_{\text{NB}}(k) - \ell_{\text{P}}\right].$$

The maximum is over 81 values of $k$ from 0.01 to 1000 (`chains.offspring_lrt`), and the $k$ that gives it is the estimate $\hat k$. If no $k$ makes our counts more likely than Poisson does, $\Lambda = 0$ and $\hat k$ is printed as `inf`. Otherwise $\Lambda > 0$: the larger it is, the more the counts are spread out (many zeros and a few large counts) beyond what Poisson allows.

### 6c. The p-value

Write $\Lambda_{\text{obs}}$ for the observed value of $\Lambda$, computed from the export this notebook read. With few vampires, no printed table gives the chances of $\Lambda$ accurately, so we get them by simulation: make `REPS` sets of $n$ counts from the Poisson model with average $R$, the observed average (`model.poisson`), compute $\Lambda$ for each set, and take the share of sets with $\Lambda$ equal to $\Lambda_{\text{obs}}$ or larger:

$$p = \Pr(\Lambda \ge \Lambda_{\text{obs}}) .$$

Unlike the shuffle, this p-value is **not** conditional on the total: each made-up set has its own total. It takes about 10 seconds for each version.

**About $\hat k$.** With few biters, $\hat k$ is very uncertain: very different values of $k$ give our counts almost the same likelihood. Report $\hat k$, and do not report an interval for it narrower than the data allow; this notebook does not compute one.

In [ ]:
# Step 6: the likelihood-ratio statistic, the estimate of k, and the p-value by simulation
rng = random.Random(6)
lrt = {}
for name, counts in versions.items():
    obs, k_hat = chains.offspring_lrt(counts)
    m = sum(counts) / n
    hits = sum(chains.offspring_lrt([poisson(m, rng) for _ in range(n)])[0] >= obs - 1e-9
               for _ in range(REPS))
    lrt[name] = (obs, k_hat, hits / REPS)
    print(f"{name}: Lambda = {obs:.2f}, k_hat = {k_hat:.2f}, "
          f"p-value (share of {REPS} Poisson sets with Lambda equal to the observed Lambda or larger) = {show_p(hits / REPS, REPS)}")

## Step 7: the figure

For each version: bars for how many vampires had each count, and the number of vampires each model expects with each count, with the average set to the observed $R$: Poisson (circles) and the negative binomial at $\hat k$ (crosses). If $\hat k$ is `inf`, the two sets of marks are the same. If vampires differ, the bars at 0 and at the largest counts stand above the circles, and the bar at 1 below them.

In [ ]:
# Step 7: the figure
import matplotlib.pyplot as plt

def nb_pmf(c, mean, k):
    if math.isinf(k) or mean == 0:
        return math.exp(-mean) * mean ** c / math.factorial(c)
    return math.exp(math.lgamma(c + k) - math.lgamma(k) - math.lgamma(c + 1)
                    + k * math.log(k / (k + mean)) + c * math.log(mean / (k + mean)))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, counts) in zip(axes, versions.items()):
    mean, k_hat = sum(counts) / n, lrt[name][1]
    xs = list(range(max(counts) + 2))
    ax.bar(xs, [counts.count(c) for c in xs], color="lightgray", edgecolor="black", label="our vampires")
    ax.plot(xs, [n * math.exp(-mean) * mean ** c / math.factorial(c) for c in xs], "o", color="black",
            label="Poisson expects")
    ax.plot(xs, [n * nb_pmf(c, mean, k_hat) for c in xs], "x", color="black", markersize=9,
            label=f"negative binomial, k = {k_hat:.2f}, expects")
    ax.set_xticks(xs)
    ax.set_xlabel("bites per vampire" if counts is y else "sittings per vampire")
    ax.set_ylabel("number of vampires")
    ax.set_title(name)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Step 8: what the tests do not say

- **Why vampires differ.** A small p-value says the counts are more uneven than one rate for everyone allows. It does not say whether some people bite more, or some had more chances to, or the vampires recruited in person by a walker were different from the rest. Questions 2 and 3 each test one possible reason.
- **What counts as one bite.** The two versions can disagree: a sitting of three friends is three bites by one vampire, or one sitting. If one version rejects the Poisson model and the other does not, the answer depends on what counts as one bite. A report gives both results, says which definition gives which, and says which definition it thinks fits the question, and why.
- **That vampires are alike.** A p-value of 0.05 or more says the counts are consistent with one rate for every vampire, not that vampires are alike: with few biters, the tests may not be able to tell.

## Step 9: your result

Fill this in, in your own words, in a text cell below:

- The number of finished vampires $n$, the number of bites $B$, the number of sittings $S$ and `SITTING_MINUTES`, and the number of biters $D$.
- $R$, with its interval (Step 4).
- For each version: the shuffle test's p-value (Step 5) and whether the simulation agrees, and $\Lambda$, $\hat k$, and the likelihood-ratio p-value (Step 6).
- What your plan said each result would mean, and which one happened.
- What the result does not say (Step 8).